# Taller 3: Pipeline ETL con Apache Kafka + Aprendizaje Automático
**Curso:** ETL (G01) · Ingeniería de Datos e Inteligencia Artificial

**Objetivo:** consumir los registros del World Happiness Report desde Kafka, construir un dataset unificado, hacer EDA y entrenar/evaluar un modelo de regresión que prediga el *happiness score*.

**Flujo (según el diagrama de bloques):**
`Kafka (5 CSV) → Consumidor → DataFrame en memoria → Armonización + EDA → Selección de características → Entrenamiento → Evaluación → Predicción → Carga a base de datos (SQLite)`

**Ejecución:** en Colab, `Entorno de ejecución → Ejecutar todo`. Se pedirá la contraseña de Kafka (la da el instructor).

## 0. Instalación e importaciones

In [ ]:
!pip install -q confluent-kafka certifi pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
import json, re, sqlite3, getpass, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import certifi, joblib
from confluent_kafka import Consumer, KafkaError

from sklearn.model_selection import train_test_split, GroupShuffleSplit, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Extracción de datos (Kafka)
### 1.1 Configuración y consumidor
Se usa `auto.offset.reset = earliest` para leer el tema desde el inicio y `enable.auto.commit = False` para que cada ejecución pueda releer todo. El `group.id` debe ser **único por estudiante** (cámbialo por el tuyo) para no compartir offsets con otros.

In [ ]:
KAFKA_PASSWORD = getpass.getpass("Contraseña SASL (la da el instructor): ")
TOPIC = "etl.records"
GROUP_ID = "student-yuraniduque"      # <-- tu group.id (único por estudiante)

conf = {
    "bootstrap.servers": "kafka-classroom-2026.westus.cloudapp.azure.com:9092",
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "SCRAM-SHA-512",
    "sasl.username": "classroom-students",
    "sasl.password": KAFKA_PASSWORD,
    "ssl.ca.location": certifi.where(),
    "group.id": GROUP_ID,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
}

### 1.2 Consumo y acumulación en memoria
Se hace `poll()` hasta que el tema deja de entregar mensajes (varios polls vacíos seguidos). Cada mensaje se guarda con sus **metadatos** (tema, partición, offset, timestamp, key, headers) y su **payload** original. Se evitan duplicados por `(partición, offset)`.

In [ ]:
MAX_EMPTY_POLLS = 10     # polls vacíos consecutivos antes de dar por terminado el consumo

def decode_payload(raw):
    if raw is None:
        return None
    text = raw.decode("utf-8", errors="replace")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text            # si no es JSON, se guarda el texto crudo

def consume_all(conf, topic, max_empty=MAX_EMPTY_POLLS):
    consumer = Consumer(conf)
    consumer.subscribe([topic])
    records, seen, empty = [], set(), 0
    try:
        while empty < max_empty:
            msg = consumer.poll(1.0)
            if msg is None:
                empty += 1
                continue
            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    empty += 1
                    continue
                raise RuntimeError(msg.error())
            empty = 0
            uid = (msg.partition(), msg.offset())
            if uid in seen:                       # deduplicación
                continue
            seen.add(uid)
            records.append({
                "_topic": msg.topic(),
                "_partition": msg.partition(),
                "_offset": msg.offset(),
                "_timestamp": msg.timestamp()[1],
                "_key": msg.key().decode("utf-8", errors="replace") if msg.key() else None,
                "_headers": {k: (v.decode("utf-8", errors="replace") if v else None)
                             for k, v in (msg.headers() or [])},
                "payload": decode_payload(msg.value()),
            })
    finally:
        consumer.close()
    return records

records = consume_all(conf, TOPIC)
print(f"Mensajes consumidos: {len(records)}")

### 1.3 Inspección de la estructura del mensaje

In [ ]:
for r in records[:3]:
    print(json.dumps(r, indent=2, ensure_ascii=False, default=str))
    print("-" * 60)

print("Tipos de payload:", pd.Series([type(r["payload"]).__name__ for r in records]).value_counts().to_dict())
print("Particiones:", sorted({r["_partition"] for r in records}))

### 1.3b Estado de la aplicación
Como en la clase de aplicaciones en tiempo real: lo consumido se acumula en un **estado** (memoria de la aplicación), no en variables sueltas. Aquí se descartan además los duplicados por `event_id` (si el payload lo trae), porque la entrega de Kafka es *al menos una vez*.

In [ ]:
seen_ids, unique = set(), []
for r in records:
    pl = r["payload"] if isinstance(r["payload"], dict) else {}
    eid = pl.get("event_id") or (pl.get("metadata") or {}).get("record_id")   # p. ej. "2015-4"
    if eid is not None and eid in seen_ids:
        continue
    if eid is not None:
        seen_ids.add(eid)
    unique.append(r)

state = {
    "events": unique,
    "last_event": unique[-1] if unique else None,
    "by_partition": pd.Series([r["_partition"] for r in unique]).value_counts().to_dict() if unique else {},
    "duplicates_by_event_id": len(records) - len(unique),
}
records = unique
print({k: v for k, v in state.items() if k not in ("events", "last_event")})
print("Eventos únicos en el estado:", len(state["events"]))

### 1.4 Construcción del DataFrame crudo
Se aplana el payload (`json_normalize`) y se conservan los metadatos con prefijo `_`.
**Año:** se busca primero un campo `year` en el payload; si no existe, se intenta extraerlo de la key, los headers o un campo con el nombre del archivo. *Revisa la salida de 1.3 y ajusta `YEAR_CANDIDATES` si tu mensaje usa otro nombre.*

In [ ]:
meta = pd.DataFrame([{k: v for k, v in r.items() if k != "payload"} for r in records])

# Formato real del tema: {"metadata": {source_file, year, record_id}, "data": {...columnas del CSV...}}
def unwrap(payload):
    if isinstance(payload, dict) and isinstance(payload.get("data"), dict):
        return dict(payload["data"]), dict(payload.get("metadata") or {})
    return (dict(payload) if isinstance(payload, dict) else {"value": payload}), {}

pairs = [unwrap(r["payload"]) for r in records]
datas, metas = [d for d, _ in pairs], [m for _, m in pairs]

body = pd.json_normalize(datas)
msg_meta = pd.DataFrame([{f"_{k}": v for k, v in m.items() if k != "year"} for m in metas])   # _source_file, _record_id
raw_df = pd.concat([meta.drop(columns=["_headers"]), msg_meta, body], axis=1)

YEAR_CANDIDATES = ["year", "Year", "anio", "año", "source_year", "file", "source_file", "filename"]

def find_year(fields, key, headers):
    for c in YEAR_CANDIDATES:
        if fields.get(c) is not None:
            m = re.search(r"(20\d{2})", str(fields[c]))
            if m: return int(m.group(1))
    for src in [key] + list((headers or {}).values()):
        m = re.search(r"(20\d{2})", str(src))
        if m: return int(m.group(1))
    return np.nan

raw_df["year"] = [find_year({**metas[i], **datas[i]}, records[i]["_key"], records[i]["_headers"])
                  for i in range(len(raw_df))]
print("Forma:", raw_df.shape)
print("Registros por año:\n", raw_df["year"].value_counts(dropna=False).sort_index())
raw_df.head()


## 2. Preparación de datos
### 2.1 Columnas por año: comunes vs. únicas
Cada año del World Happiness Report usa nombres distintos (p. ej. `Happiness Score`, `Happiness.Score`, `Score`, `Ladder score`), así que primero se compara qué columnas trae cada año.

In [ ]:
meta_cols = [c for c in raw_df.columns if c.startswith("_")]
cols_by_year = {}
for y, g in raw_df.groupby("year"):
    cols_by_year[int(y)] = set(g.drop(columns=meta_cols + ["year"]).dropna(axis=1, how="all").columns)

common = set.intersection(*cols_by_year.values())
print("Columnas comunes a TODOS los años (nombre original):", sorted(common), "\n")
for y, cols in sorted(cols_by_year.items()):
    others = set.union(*[c for yy, c in cols_by_year.items() if yy != y])
    print(f"{y}: {len(cols)} columnas | únicas de este año: {sorted(cols - others)}")
    print("    ", sorted(cols))

### 2.2 Armonización de nombres
Se normalizan los nombres (minúsculas, sin símbolos) y se mapean con reglas a un esquema común. **Nota de calidad:** `Family` (2015–2017) y `Social support` (2018+) miden un concepto muy similar pero **no idéntico**; aquí se tratan como la misma variable y se documenta como limitación.

In [ ]:
def norm(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")

RULES = [  # (regex, nombre final) — el orden importa
    (r"^(country|country_or_region|country_name)$", "country"),
    (r"^region$", "region"),
    (r"lower_confidence|upper_confidence|whisker|standard_error|confidence", "ci_or_se"),
    (r"dystopia", "dystopia_residual"),
    (r"^(happiness_score|score|ladder_score)$", "happiness_score"),
    (r"rank", "happiness_rank"),
    (r"gdp|economy", "gdp_per_capita"),
    (r"social_support|^family$", "social_support"),
    (r"health|life_expectancy", "life_expectancy"),
    (r"^freedom", "freedom"),
    (r"corruption|trust", "corruption"),
    (r"generosity", "generosity"),
]

def map_col(c):
    n = norm(c)
    for pat, target in RULES:
        if re.search(pat, n):
            return target
    return n

def harmonize(df):
    out = []
    for y, g in df.groupby("year"):
        g = g.drop(columns=[c for c in g.columns if c.startswith("_")]).dropna(axis=1, how="all").copy()
        g.columns = [map_col(c) if c != "year" else "year" for c in g.columns]
        g = g.loc[:, ~g.columns.duplicated()]          # por si dos columnas caen en el mismo destino
        out.append(g)
    return pd.concat(out, ignore_index=True, sort=False)

df = harmonize(raw_df)
print(df.shape)
print(sorted(df.columns))
df.head()

### 2.3 Calidad de datos: tipos, faltantes, duplicados, formatos y atípicos

In [ ]:
NUMERIC = ["happiness_score", "gdp_per_capita", "social_support", "life_expectancy",
           "freedom", "corruption", "generosity", "dystopia_residual", "happiness_rank"]
for c in NUMERIC:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")      # formatos inconsistentes -> NaN
df["year"] = df["year"].astype("Int64")

# Formato inconsistente en nombres de país
df["country"] = df["country"].astype(str).str.strip()
COUNTRY_FIX = {
    "Taiwan Province of China": "Taiwan", "Hong Kong S.A.R., China": "Hong Kong",
    "Trinidad & Tobago": "Trinidad and Tobago", "North Cyprus": "Northern Cyprus",
    "Macedonia": "North Macedonia", "Somaliland region": "Somaliland Region",
    "Congo (Kinshasa)": "Democratic Republic of the Congo", "Congo (Brazzaville)": "Congo",
}
df["country"] = df["country"].replace(COUNTRY_FIX)

print("== Valores faltantes por columna ==")
print(df.isna().sum()[df.isna().sum() > 0])
print("\n== Faltantes por año ==")
print(df.groupby("year").apply(lambda g: g.isna().sum()[g.isna().sum() > 0]))
dups = df.duplicated(subset=["country", "year"]).sum()
print(f"\nDuplicados (country, year): {dups}")
df = df.drop_duplicates(subset=["country", "year"]).reset_index(drop=True)

# Rangos esperados
print("\n== Resumen numérico ==")
display(df[[c for c in NUMERIC if c in df.columns]].describe().T)

In [ ]:
# Atípicos con regla IQR (solo se marcan; no se eliminan: en este dominio un país extremo es un dato real)
feat_cols = ["happiness_score", "gdp_per_capita", "social_support", "life_expectancy",
             "freedom", "corruption", "generosity"]
rows = []
for c in feat_cols:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    mask = (df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)
    rows.append({"variable": c, "n_atipicos": int(mask.sum()), "pct": round(100 * mask.mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
df[feat_cols[1:]].plot(kind="box", ax=ax); ax.set_title("Boxplots de variables predictoras"); plt.show()

### 2.4 Integración de los años
**Decisión:** apilar los años en formato **largo** (una fila = país-año) con la columna `year`, conservando solo el esquema común.

**Justificación:** triplica/quintuplica la muestra respecto a un solo año, conserva la variación temporal y las variables comunes son justo los factores del reporte.

**Limitaciones:**
- Las observaciones del mismo país en distintos años están correlacionadas → un split aleatorio puede filtrar información entre train y test (se contrasta abajo con un split por país).
- `Family` vs `Social support`, y cambios de metodología del reporte entre años, introducen cierta inconsistencia.
- Se **excluyen** `happiness_rank` y `dystopia_residual`: el primero es función directa de la variable objetivo y el segundo se calcula a partir de ella (*data leakage*).
- Los faltantes (p. ej. `corruption`) se imputan **dentro del pipeline** con datos de entrenamiento únicamente.

In [ ]:
FEATURES = ["gdp_per_capita", "social_support", "life_expectancy", "freedom", "corruption", "generosity"]
TARGET = "happiness_score"

data = df[["country", "year", TARGET] + FEATURES].dropna(subset=[TARGET]).reset_index(drop=True)
print("Dataset final:", data.shape)
print(data.groupby("year").size())

## 3. Análisis exploratorio (EDA)

In [ ]:
display(data.groupby("year")[[TARGET] + FEATURES].mean().round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data[TARGET], kde=True, ax=axes[0]); axes[0].set_title("Distribución del happiness score")
sns.boxplot(data=data, x="year", y=TARGET, ax=axes[1]); axes[1].set_title("Score por año")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(data[[TARGET] + FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación"); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), FEATURES):
    sns.scatterplot(data=data, x=f, y=TARGET, hue="year", palette="viridis", alpha=.6, ax=ax, legend=(f == FEATURES[0]))
    ax.set_title(f"{f} vs {TARGET}")
plt.tight_layout(); plt.show()

### Resumen de hallazgos del EDA
*(Complétalo con tus propios resultados al ejecutar; guía:)*
- Variables más correlacionadas con el score: normalmente **PIB per cápita, apoyo social y esperanza de vida** (correlaciones altas y positivas).
- `generosity` y `corruption` suelen mostrar relación más débil; `corruption` es la que más faltantes y atípicos presenta.
- Hay diferencias de nombres y escalas entre años (ver 2.1 y 2.2) y algunos países cambian de nombre entre reportes.
- No se eliminaron atípicos porque representan países reales; los modelos basados en árboles son robustos a ellos.

## 4. Entrenamiento del modelo
- **Objetivo:** `happiness_score` · **Predictoras:** las 6 variables seleccionadas.
- **Split 80/20.** Todo el preprocesamiento (imputación + estandarización) vive dentro de un `Pipeline` que se ajusta **solo con train**, evitando fuga de información desde test.
- Se compara regresión lineal (baseline interpretable) con Ridge, Random Forest y Gradient Boosting usando validación cruzada **solo sobre train**.

In [ ]:
X, y, groups = data[FEATURES], data[TARGET], data["country"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

def make_pipe(model, scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale: steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

models = {
    "LinearRegression": make_pipe(LinearRegression()),
    "Ridge": make_pipe(Ridge(alpha=1.0)),
    "RandomForest": make_pipe(RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1), scale=False),
    "GradientBoosting": make_pipe(GradientBoostingRegressor(random_state=RANDOM_STATE), scale=False),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, pipe in models.items():
    s = -cross_val_score(pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    cv_rows.append({"modelo": name, "RMSE_cv_medio": s.mean(), "RMSE_cv_std": s.std()})
cv_df = pd.DataFrame(cv_rows).sort_values("RMSE_cv_medio")
display(cv_df.round(4))

## 5. Evaluación

In [ ]:
def adj_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

def evaluate(pipe, Xtr, ytr, Xte, yte):
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    mse = mean_squared_error(yte, pred)
    r2 = r2_score(yte, pred)
    return pred, {"MAE": mean_absolute_error(yte, pred), "MSE": mse, "RMSE": np.sqrt(mse),
                  "R2": r2, "R2_ajustado": adj_r2(r2, len(yte), Xte.shape[1])}

results, preds = [], {}
for name, pipe in models.items():
    preds[name], m = evaluate(pipe, X_train, y_train, X_test, y_test)
    results.append({"modelo": name, **m})
res_df = pd.DataFrame(results).sort_values("RMSE")
display(res_df.round(4))

best_name = cv_df.iloc[0]["modelo"]          # se elige con validación cruzada (train), NO con test
best_pipe = models[best_name]
print("Mejor modelo (validación cruzada):", best_name)

### 5.1 Chequeo de robustez: split por país
Con un split aleatorio, el mismo país puede aparecer en train y test en años distintos. Aquí se separa **por país** para estimar el desempeño en países *no vistos*.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X, y, groups))
_, m_group = evaluate(make_pipe(LinearRegression()), X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[te_idx], y.iloc[te_idx])
print("LinearRegression con split por país:", {k: round(v, 4) for k, v in m_group.items()})

In [ ]:
pred = preds[best_name]
resid = y_test - pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].scatter(y_test, pred, alpha=.6)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "r--"); axes[0].set_xlabel("Real"); axes[0].set_ylabel("Predicho"); axes[0].set_title(f"Real vs predicho ({best_name})")
axes[1].scatter(pred, resid, alpha=.6); axes[1].axhline(0, color="r", ls="--")
axes[1].set_xlabel("Predicho"); axes[1].set_ylabel("Residuo"); axes[1].set_title("Residuos vs predicho")
sns.histplot(resid, kde=True, ax=axes[2]); axes[2].set_title("Distribución de residuos")
plt.tight_layout(); plt.show()

imp = permutation_importance(best_pipe, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE, scoring="r2")
imp_s = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
imp_s.plot(kind="barh", figsize=(7, 3.5), title="Importancia por permutación (caída de R²)"); plt.show()

lin = models["LinearRegression"]
coefs = pd.Series(lin.named_steps["model"].coef_, index=FEATURES).sort_values()
coefs.plot(kind="barh", figsize=(7, 3.5), title="Coeficientes (regresión lineal, variables estandarizadas)"); plt.show()

### 5.2 Interpretación y mejoras
- **MAE / RMSE** están en puntos de la escala de felicidad (0–10): un RMSE de ~0.5 significa que, en promedio, la predicción se desvía ~0.5 puntos. **R²** indica la proporción de varianza explicada; el **R² ajustado** penaliza el número de predictoras.
- Compara la regresión lineal con los modelos de árboles: si la ganancia es pequeña, la lineal es preferible por interpretabilidad.
- Si el rendimiento cae con el split por país, el modelo depende parcialmente de "conocer" al país en otros años.
- **Posibles mejoras:** variables categóricas (región), incluir `year`, ajuste de hiperparámetros (`GridSearchCV`), transformaciones no lineales, tratamiento específico de faltantes en `corruption`, y validación temporal (entrenar con años previos y probar en el último).

## 6. Persistencia: modelo y carga a base de datos
Cierra el flujo del diagrama (*Model prediction → Database load*). Se guarda el modelo y se cargan datos limpios y predicciones en **SQLite**.

In [ ]:
final_model = make_pipe(best_pipe.named_steps["model"].__class__(**best_pipe.named_steps["model"].get_params()),
                        scale="scaler" in best_pipe.named_steps)
final_model.fit(X_train, y_train)
joblib.dump(final_model, "happiness_model.joblib")

def predict_record(record: dict):
    row = pd.DataFrame([{f: record.get(f) for f in FEATURES}])
    return float(final_model.predict(row)[0])

con = sqlite3.connect("happiness.db")
data.to_sql("happiness_clean", con, if_exists="replace", index=False)

test_out = data.loc[X_test.index, ["country", "year", TARGET]].copy()
test_out["predicted_score"] = final_model.predict(X_test)
test_out["abs_error"] = (test_out[TARGET] - test_out["predicted_score"]).abs()
test_out.rename(columns={TARGET: "actual_score"}).to_sql("happiness_predictions", con, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS n FROM happiness_predictions", con))
display(pd.read_sql("SELECT * FROM happiness_predictions ORDER BY abs_error DESC LIMIT 5", con))
con.close()

### 6.1 Metadatos del modelo e inferencia con un evento nuevo
Igual que en la práctica de ML: el modelo ya está entrenado, aquí **no se vuelve a llamar a `fit()`**. Se guarda un archivo de metadatos junto al modelo y se valida el evento antes de predecir.

In [ ]:
import platform, sklearn
from datetime import datetime, timezone

metadata = {
    "model": best_name, "target": TARGET, "features": FEATURES,
    "n_train": len(X_train), "n_test": len(X_test),
    "test_metrics": res_df[res_df["modelo"] == best_name].iloc[0][["MAE", "MSE", "RMSE", "R2", "R2_ajustado"]].astype(float).to_dict(),
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "sklearn_version": sklearn.__version__, "python_version": platform.python_version(),
}
with open("happiness_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

loaded = joblib.load("happiness_model.joblib")

def validate_event(ev: dict):
    vals = [ev.get(f) for f in FEATURES]
    if sum(v is not None for v in vals) < 4:
        return "faltan predictoras"
    if any(v is not None and not (0 <= v <= 3) for v in vals):
        return "predictora fuera de rango"
    return None

nuevo = {"gdp_per_capita": 1.2, "social_support": 1.3, "life_expectancy": 0.9,
         "freedom": 0.5, "corruption": 0.1, "generosity": 0.2}
problema = validate_event(nuevo)
if problema:
    print("Evento rechazado:", problema)
else:
    pred = loaded.predict(pd.DataFrame([nuevo]))[0]
    print(f"Happiness score predicho: {pred:.3f}")

## 7. Documentación técnica
| Etapa | Decisión |
|---|---|
| Kafka | `confluent-kafka`, SASL_SSL + SCRAM-SHA-512, tema `etl.records`, `earliest`, dedup por (partición, offset), metadatos conservados |
| Estado en memoria | `state` (eventos únicos, último evento, conteo por partición, duplicados) → `DataFrame` con metadatos (`_partition`, `_offset`, `_timestamp`, …) |
| Fusión | reglas regex para unificar nombres, formato largo con `year`, esquema común |
| Calidad | coerción numérica, limpieza de nombres de país, deduplicación, atípicos marcados (IQR) |
| Fuga de información | se excluyen `rank` y `dystopia_residual`; preprocesamiento dentro del `Pipeline` |
| Modelos | Lineal, Ridge, Random Forest, Gradient Boosting; CV 5 folds sobre train |
| Métricas | MAE, MSE, RMSE, R², R² ajustado + chequeo con split por país |
| Salida | `happiness_model.joblib`, `happiness.db` (SQLite) |